In [0]:
# analysis/spark_analysis.py
s3_ext_location = 's3://awae-yellow-taxi-case/'

from pyspark.sql import functions as F

print("--- Starting Spark Analytical Notebook ---")

In [0]:
# Load the curated fact table from Gold layer
path_gold = f"{s3_ext_location}/gold/taxi_trips_curated/"
df_gold = spark.read.format("parquet").load(path_gold)

# Register the DataFrame as a temporary SQL view to allow SparkSQL queries
df_gold.createOrReplaceTempView("vw_taxi_trips_curated")

## APPROACH A: ANSWERING QUESTIONS USING PYSPARK DATAFRAMES

### Q01: Qual a média de valor total (total\_amount) recebido em um mês considerando todos os yellow táxis da frota?

In [0]:
# Create a reusable filtered data frame only with data for yellow cabs
df_yellow = df_gold.filter(F.col("taxi_type") == "yellow")

Note: As I was in doubt about the question (in withc month? is the agrgregate of all data over the months? it´s by year/month? - I decided to follow with 3 possible answers)

Response 1: Average ticket per specific Year/Month block

In [0]:
df_q2_r1_pyspark = df_yellow.groupBy("pickup_year", "pickup_month") \
                            .agg(F.round(F.avg("total_amount"), 2).alias("average_ticket_value")) \
                            .sort("pickup_year", "pickup_month")
display(df_q2_r1_pyspark)

Response 2: Average ticket grouped strictly by Month Part

In [0]:
df_q2_r2_pyspark = df_yellow.groupBy("pickup_month") \
                            .agg(F.round(F.avg("total_amount"), 2).alias("average_ticket_for_this_month")) \
                            .sort("pickup_month")

display(df_q2_r2_pyspark)

Response 3:Global Average Ticket Baseline (Single Value)

In [0]:
df_q2_r3_pyspark = df_yellow.agg(F.round(F.avg("total_amount"), 2).alias("global_average_ticket"))
display(df_q2_r3_pyspark)

### Q02: Qual a média de passageiros (passenger\_count) por cada hora do dia que pegaram táxi no mês de maio considerando todos os táxis da frota?

Response: Average passengers (passenger_count) per hour in May (all Fleet ==> Yellow and Green Taxi)

In [0]:
df_q3_pyspark = df_gold.filter((F.col("pickup_year") == 2023) & (F.col("pickup_month") == "05")) \
                       .groupBy("pickup_hour") \
                       .agg(F.round(F.avg("passenger_count"), 2).alias("avg_passengers")) \
                       .sort("pickup_hour")
display(df_q3_pyspark)

## APPROACH B: ANSWERING QUESTIONS USING SPARKSQL

### Q01: Qual a média de valor total (total\_amount) recebido em um mês considerando todos os yellow táxis da frota?

In [0]:
print("Response 1: Average passengers (passenger_count) per hour in May (all Fleet ==> Yellow and Green Taxi")
display(spark.sql("""
    SELECT 
        pickup_year,
        pickup_month,
        ROUND(AVG(total_amount), 2) AS average_ticket_value
    FROM vw_taxi_trips_curated
    WHERE taxi_type = 'yellow'
    GROUP BY pickup_year, pickup_month
    ORDER BY pickup_year, pickup_month
"""))

In [0]:
print("Response 2: Average ticket grouped strictly by Month Part")

display(spark.sql("""
    SELECT 
        pickup_month,
        ROUND(AVG(total_amount), 2) AS average_ticket_for_this_month
    FROM vw_taxi_trips_curated
    WHERE taxi_type = 'yellow'
    GROUP BY pickup_month
    ORDER BY pickup_month
"""))

In [0]:
print(" Response 3: Global Average Ticket Baseline (Single Value)")
display(spark.sql("""
    SELECT ROUND(AVG(total_amount), 2) AS global_average_ticket
    FROM vw_taxi_trips_curated
    WHERE taxi_type = 'yellow'
"""))

### Q02: Qual a média de passageiros (passenger\_count) por cada hora do dia que pegaram táxi no mês de maio considerando todos os táxis da frota?

In [0]:
print("Question 3: Average passenger_count per hour in May (All Fleet)")
display(spark.sql("""
    SELECT 
        pickup_hour,
        ROUND(AVG(passenger_count), 2) AS average_passengers_per_hour
    FROM vw_taxi_trips_curated
    WHERE pickup_year = 2023 AND pickup_month = '05'
    GROUP BY pickup_hour
    ORDER BY pickup_hour ASC
"""))